# RoBERTa-BNE multietiqueta — Seguridad y subcategorías


Este experimento usa exclusivamente la tarea multietiqueta de Seguridad: `Security` y sus seis
subcategorías. El protocolo de carga, validación, partición y métricas vive en
`security_multilabel_common.py`, por lo que los cuatro modelos consumen exactamente el mismo
manifiesto `artifacts/security_split_manifest_seed42.csv`.

El CSV actual contiene preanotaciones automáticas. `ALLOW_PROVISIONAL_LABELS=True` permite este
prototipo; cambiarlo a `False` exige `expert_validated` o `consensus` en todos los positivos.
Test está reservado y no interviene en entrenamiento, selección de configuración ni umbral.

Requisitos: `pip install -r requirements-security.txt`.


## 1. Configuración y reproducibilidad


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

from security_multilabel_common import (
    DATA_PATH, LABEL_COLS, SEED, SPLIT_PATH, TEXT_COL,
    get_or_create_splits, is_better, load_security_dataset,
    multilabel_metrics, prediction_frame, save_json, seed_everything, select_per_label_thresholds,
)

ALLOW_PROVISIONAL_LABELS = True
USE_TRAIN_AUGMENTATION = False  # activar solo después de revisión experta
USE_MODERATE_POS_WEIGHT = False  # comparar como ablación tras corregir etiquetas
RUN_HPARAM_SEARCH = False
RUN_FINAL_TEST = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| PyTorch:", torch.__version__)


In [ ]:
MODEL_NAME = "PlanTL-GOB-ES/roberta-base-bne"

# El repositorio primario fue deprecado y actualmente solo publica README.md.
# Este mirror no oficial declara haber recuperado los pesos originales. El uso queda visible
# en los artefactos; poner False para abortar en vez de usar el mirror.
MODEL_MIRROR = "Buzzeitor/roberta-base-bne"
ALLOW_UNOFFICIAL_MIRROR = True
from huggingface_hub import list_repo_files
primary_files = set(list_repo_files(MODEL_NAME))
if "config.json" in primary_files:
    MODEL_SOURCE = MODEL_NAME
elif ALLOW_UNOFFICIAL_MIRROR:
    MODEL_SOURCE = MODEL_MIRROR
    print(f"AVISO: {MODEL_NAME} no publica pesos; se usará el mirror explícito {MODEL_SOURCE}.")
else:
    raise RuntimeError(
        f"{MODEL_NAME} no publica actualmente config/tokenizer/pesos. "
        "Active ALLOW_UNOFFICIAL_MIRROR o proporcione una copia local verificada."
    )

ROBERTA_CONFIG = {
    "learning_rate": 2e-5, "batch_size": 16, "epochs": 5, "dropout": 0.1,
    "weight_decay": 0.01, "max_length": 128, "optimizer": "AdamW", "threshold": 0.50,
}
INITIAL_CONFIG = ROBERTA_CONFIG.copy()
SEARCH_SPACE = {
    "learning_rate": [1e-5, 5e-5], "batch_size": [8, 16], "epochs": [5, 10, 15, 20],
    "dropout": [0.1, 0.5], "weight_decay": [1e-8, 0.3], "max_length": [128, 256],
    "optimizer": ["AdamW", "Adam"], "threshold": [0.40, 0.60],
}
SEARCH_TRIALS = [
    INITIAL_CONFIG.copy(),
    {**INITIAL_CONFIG, "learning_rate": 1e-5, "batch_size": 8, "epochs": 10,
     "dropout": 0.2, "weight_decay": 1e-4, "max_length": 256, "threshold": 0.45},
    {**INITIAL_CONFIG, "learning_rate": 5e-5, "epochs": 15, "dropout": 0.5,
     "weight_decay": 0.1, "threshold": 0.55},
]
MODEL_KEY = "roberta_bne_security"
CHECKPOINT_DIR = Path("checkpoints/roberta_bne_security")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR, LAST_DIR = CHECKPOINT_DIR / "best", CHECKPOINT_DIR / "last"
HISTORY_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_history.csv"
TRIALS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_trials.csv"
VAL_PRED_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_validation_predictions.csv"
VAL_LABEL_METRICS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_validation_metrics_by_label.csv"
WINNER_CONFIG_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_winning_config.json"
CURVES_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_curves.png"
TEST_PRED_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_test_predictions.csv"
TEST_LABEL_METRICS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_test_metrics_by_label.csv"
TEST_METRICS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_test_metrics.json"


## 2. Dataset validado y partición común


In [ ]:
df = load_security_dataset(DATA_PATH, allow_provisional_labels=ALLOW_PROVISIONAL_LABELS)
train_df, val_df, test_df, split_manifest = get_or_create_splits(df, SPLIT_PATH)

print("Etiquetas (orden fijo):", LABEL_COLS)
print("Particiones:", {"train": len(train_df), "val": len(val_df), "test": len(test_df)})
print("Método de estratificación:", split_manifest["stratification_method"].iloc[0])
if "fallback" in split_manifest["stratification_method"].iloc[0]:
    print("AVISO: no está instalado iterative-stratification; se usó el fallback determinista, menos fiel.")
for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(name, part[LABEL_COLS].mean().round(3).to_dict())
assert not split_manifest.query("split in ['val', 'test']")["is_synthetic"].astype(bool).any()


# El aumento queda pausado hasta completar la revisión experta de etiquetas raras.
from security_train_augmentation import augment_balanced_subcategories
real_train_df = train_df.copy()
if USE_TRAIN_AUGMENTATION:
    train_df = augment_balanced_subcategories(real_train_df)
else:
    train_df = real_train_df
print("Train real / utilizado:", len(real_train_df), "/", len(train_df))
print("Positivos utilizados:", train_df[LABEL_COLS].sum().astype(int).to_dict())

# Peso moderado opcional: sqrt(N_neg/N_pos), limitado a 10 y calculado solo con train.
positive_counts = train_df[LABEL_COLS].sum().to_numpy(dtype=float)
negative_counts = len(train_df) - positive_counts
moderate_pos_weight = np.minimum(np.sqrt(negative_counts / np.maximum(positive_counts, 1.0)), 10.0)
POS_WEIGHT = torch.tensor(moderate_pos_weight, dtype=torch.float32, device=DEVICE)
print("Moderate pos_weight:", dict(zip(LABEL_COLS, moderate_pos_weight.round(3))))

def make_criterion():
    return nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT if USE_MODERATE_POS_WEIGHT else None)


## 3. Tokenizador y modelo preentrenado (sin stemming ni stopwords)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)

class TransformerDataset(Dataset):
    def __init__(self, frame, max_length):
        self.frame = frame.reset_index(drop=True)
        self.max_length = max_length
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        encoded = tokenizer(str(row[TEXT_COL]), truncation=True, padding="max_length",
                            max_length=self.max_length, return_tensors="pt")
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(row[LABEL_COLS].to_numpy(dtype=np.float32))
        item["id"] = str(row["ID"])
        return item

def make_loaders(config):
    generator = torch.Generator().manual_seed(SEED)
    return (
        DataLoader(TransformerDataset(train_df, config["max_length"]), batch_size=config["batch_size"],
                   shuffle=True, generator=generator),
        DataLoader(TransformerDataset(val_df, config["max_length"]), batch_size=config["batch_size"], shuffle=False),
        DataLoader(TransformerDataset(test_df, config["max_length"]), batch_size=config["batch_size"], shuffle=False),
    )

def build_model(config, source=MODEL_SOURCE):
    hf_config = AutoConfig.from_pretrained(source, num_labels=len(LABEL_COLS),
                                            problem_type="multi_label_classification")
    # BERT/RoBERTa aceptan estos campos; classifier_dropout cubre versiones que lo priorizan.
    hf_config.hidden_dropout_prob = config["dropout"]
    hf_config.attention_probs_dropout_prob = config["dropout"]
    hf_config.classifier_dropout = config["dropout"]
    return AutoModelForSequenceClassification.from_pretrained(source, config=hf_config)


## 4. BCE multietiqueta y entrenamiento


In [ ]:
def forward_logits(model, batch):
    inputs = {key: value.to(DEVICE) for key, value in batch.items() if key not in {"id", "labels"}}
    return model(**inputs).logits

def evaluate(model, loader, threshold):
    model.eval()
    criterion = make_criterion()
    ids, truths, probabilities, total = [], [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].to(DEVICE)
            logits = forward_logits(model, batch)
            total += criterion(logits, labels).item() * len(labels)
            ids.extend(batch["id"])
            truths.append(labels.cpu().numpy())
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
    metrics = multilabel_metrics(np.vstack(truths), np.vstack(probabilities), threshold)
    metrics.update({"loss": total / len(loader.dataset), "ids": ids,
                    "y_true": np.vstack(truths), "y_prob": np.vstack(probabilities)})
    return metrics

def run_configuration(config, trial_id):
    seed_everything(SEED + trial_id)
    train_loader, val_loader, _ = make_loaders(config)
    model = build_model(config).to(DEVICE)
    criterion = make_criterion()
    optimizer_class = torch.optim.AdamW if config["optimizer"] == "AdamW" else torch.optim.Adam
    optimizer = optimizer_class(model.parameters(), lr=config["learning_rate"],
                                weight_decay=config["weight_decay"])
    trial_dir = CHECKPOINT_DIR / f"trial_{trial_id:03d}"
    best_dir, last_dir = trial_dir / "best", trial_dir / "last"
    history, best = [], None
    for epoch in range(1, config["epochs"] + 1):
        model.train(); total = 0.0
        for batch in train_loader:
            labels = batch["labels"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(forward_logits(model, batch), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item() * len(labels)
        current = evaluate(model, val_loader, config["threshold"])
        row = {"trial_id": trial_id, "epoch": epoch, "train_loss": total / len(train_loader.dataset),
               "val_loss": current["loss"], "val_macro_f1": current["macro_f1"],
               "val_security_recall": current["security_recall"], "val_micro_f1": current["micro_f1"]}
        history.append(row)
        model.save_pretrained(last_dir); tokenizer.save_pretrained(last_dir)
        save_json(last_dir / "training_state.json", {"epoch": epoch, "config": config, **row})
        if is_better(current, best):
            best = {"macro_f1": current["macro_f1"], "security_recall": current["security_recall"],
                    "loss": current["loss"], "epoch": epoch}
            model.save_pretrained(best_dir); tokenizer.save_pretrained(best_dir)
            save_json(best_dir / "metrics_config.json", {"config": config, **best, "seed": SEED,
                      "label_cols": LABEL_COLS, "model_name_requested": MODEL_NAME,
                      "model_source_used": MODEL_SOURCE})
        print(f"trial={trial_id} epoch={epoch}/{config['epochs']} val_macro_f1={current['macro_f1']:.4f} "
              f"security_recall={current['security_recall']:.4f}")
    return best, history, trial_dir


## 5. Trials, checkpoint ganador y validación


In [ ]:
configs = SEARCH_TRIALS if RUN_HPARAM_SEARCH else [INITIAL_CONFIG.copy()]
trial_rows, all_history, trial_dirs = [], [], {}
for trial_id, config in enumerate(configs):
    best, history, trial_dir = run_configuration(config.copy(), trial_id)
    trial_dirs[trial_id] = trial_dir
    all_history.extend(history)
    trial_rows.append({"trial_id": trial_id, **config, "val_macro_f1": best["macro_f1"],
                       "val_security_recall": best["security_recall"], "val_loss": best["loss"],
                       "best_epoch": best["epoch"]})

trials_df = pd.DataFrame(trial_rows).sort_values(
    ["val_macro_f1", "val_security_recall"], ascending=[False, False], kind="stable"
).reset_index(drop=True)
trials_df.to_csv(TRIALS_PATH, index=False, encoding="utf-8-sig")
history_df = pd.DataFrame(all_history)
history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
winner = trials_df.iloc[0].to_dict(); winner_id = int(winner["trial_id"])
winning_config = SEARCH_TRIALS[winner_id].copy() if RUN_HPARAM_SEARCH else INITIAL_CONFIG.copy()

# Publicar best/last del trial ganador con save_pretrained (sin tocar test).
model = AutoModelForSequenceClassification.from_pretrained(trial_dirs[winner_id] / "best").to(DEVICE)
model.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
last_model = AutoModelForSequenceClassification.from_pretrained(trial_dirs[winner_id] / "last")
last_model.save_pretrained(LAST_DIR); tokenizer.save_pretrained(LAST_DIR)
del last_model
_, val_loader, test_loader = make_loaders(winning_config)
val_metrics = evaluate(model, val_loader, winning_config["threshold"])

# Ajustar un umbral por etiqueta usando solo validation (test continúa reservado).
SELECTED_THRESHOLDS, threshold_df = select_per_label_thresholds(
    val_metrics["y_true"], val_metrics["y_prob"]
)
winning_config = {
    **winning_config,
    "thresholds_by_label": dict(zip(LABEL_COLS, SELECTED_THRESHOLDS.tolist())),
}
selected = multilabel_metrics(val_metrics["y_true"], val_metrics["y_prob"], SELECTED_THRESHOLDS)
selected.update({key: val_metrics[key] for key in ["loss", "ids", "y_true", "y_prob"]})
val_metrics = selected
threshold_df.to_csv(
    ARTIFACTS_DIR / f"{MODEL_KEY}_threshold_search.csv", index=False, encoding="utf-8-sig"
)
print("Umbrales seleccionados solo con validation:", winning_config["thresholds_by_label"])

prediction_frame(val_metrics["ids"], val_metrics).to_csv(VAL_PRED_PATH, index=False, encoding="utf-8-sig")
val_metrics["per_label"].to_csv(VAL_LABEL_METRICS_PATH, index=False, encoding="utf-8-sig")
save_json(BEST_DIR / "metrics_config.json", {"config": winning_config, "best_epoch": int(winner["best_epoch"]),
          "val_macro_f1": val_metrics["macro_f1"], "val_security_recall": val_metrics["security_recall"],
          "label_cols": LABEL_COLS, "seed": SEED, "model_name_requested": MODEL_NAME,
          "model_source_used": MODEL_SOURCE})
save_json(WINNER_CONFIG_PATH, {"config": winning_config, "best_epoch": int(winner["best_epoch"]),
          "val_macro_f1": val_metrics["macro_f1"], "val_security_recall": val_metrics["security_recall"]})
display(trials_df); display(val_metrics["per_label"])
print({key: val_metrics[key] for key in ["macro_f1", "security_recall", "micro_f1", "subset_accuracy", "hamming_loss"]})


## 6. Curvas


In [ ]:
winner_history = history_df[history_df["trial_id"].eq(winner_id)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(winner_history["epoch"], winner_history["train_loss"], label="train_loss")
axes[0].plot(winner_history["epoch"], winner_history["val_loss"], label="val_loss")
axes[1].plot(winner_history["epoch"], winner_history["val_macro_f1"], label="val_macro_f1")
axes[1].plot(winner_history["epoch"], winner_history["val_security_recall"], label="val_security_recall")
for axis in axes:
    axis.grid(alpha=.3); axis.legend(); axis.set_xlabel("Época")
fig.tight_layout()
fig.savefig(CURVES_PATH, dpi=150, bbox_inches="tight")
plt.show()


## 7. Inferencia y test reservado


In [ ]:
def predict_text(text):
    encoded = tokenizer(text, truncation=True, padding=True, max_length=winning_config["max_length"],
                        return_tensors="pt").to(DEVICE)
    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(**encoded).logits).squeeze(0).cpu().numpy()
    thresholds = np.array([winning_config.get("thresholds_by_label", {}).get(label, winning_config["threshold"]) for label in LABEL_COLS])
    return pd.DataFrame({"label": LABEL_COLS, "probability": probabilities,
                         "predicted": (probabilities >= thresholds).astype(int), "threshold": thresholds})

display(predict_text("El sistema deberá autenticar al usuario antes de permitir el acceso."))

if RUN_FINAL_TEST:
    test_metrics = evaluate(model, test_loader, SELECTED_THRESHOLDS)
    prediction_frame(test_metrics["ids"], test_metrics).to_csv(TEST_PRED_PATH, index=False, encoding="utf-8-sig")
    test_metrics["per_label"].to_csv(TEST_LABEL_METRICS_PATH, index=False, encoding="utf-8-sig")
    save_json(TEST_METRICS_PATH, {key: test_metrics[key] for key in
              ["macro_f1", "security_recall", "micro_f1", "subset_accuracy", "hamming_loss", "loss", "threshold"]})
    display(test_metrics["per_label"])
else:
    print("Test reservado: RUN_FINAL_TEST = False")
